bronze_df = spark.table(bronze_table) is logical handoff between Bronze and Silver

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

bronze_table = "workspace.incremental_project.orders_bronze_v3"
silver_table = "workspace.incremental_project.orders_silver_step4"

bronze_df = spark.table(bronze_table)

F.expr("try_cast(order_amount AS DECIMAL(12,2))") safely converts source amount while allowing malformed values to become null for validation
.filter(...) applies our simple data quality rules before MERGE

In [ ]:
clean_df = (
    bronze_df
    .withColumn(
        "order_amount",
        F.expr("try_cast(order_amount AS DECIMAL(12,2))")
    )
    .withColumn(
        "updated_at",
        F.expr("try_cast(updated_at AS TIMESTAMP)")
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("order_status").isNotNull()
        & F.col("order_amount").isNotNull()
        & (F.col("order_amount") >= 0)
        & F.col("updated_at").isNotNull()
        & F.col("_rescued_data").isNull()
    )
)

Window.paritionBy("order_id") group versions of an order
updated_at.desc() newest source update first
row_number().over(...) rank each order version
row_num == 1 select the latest version

In [ ]:
order_window = (
    Window.partitionBy("order_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("_ingested_at").desc(),
        F.col("_source_file").desc()
    )
)

latest_orders = (
    clean_df
    .withColumn(
        "row_num",
        F.row_number().over(order_window)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

merge_source = latest_orders.select(*target_columns) ensures the MERGE source contains columns expected by our Silver target
.merge(merge_source.alias("source"), "target.order_id = source.order_id") matches records by their business key
.whenMatchedUpdateAll(condition = "source.updated_at>target.updated_at") updates existing orders when incoming version is newer
.whenNotMatchedInsertAll() adds orders that aren't already in Silver. The business-key match and timestamp condition make the operation repeatable for our controlled dataset

In [ ]:
assert spark.catalog.tableExists(silver_table), (
    "Expected the existing Step 4 Silver table"
)

target_df = spark.table(silver_table)

target_columns = target_df.columns

missing_columns = [
    c for c in target_columns
    if c not in latest_orders.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

merge_source = latest_orders.select(*target_columns)

target = DeltaTable.forName(spark, silver_table)

(
    target.alias("target")
    .merge(
        merge_source.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll(
        condition = "source.updated_at > target.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver MERGE completed")

In [ ]:
silver_df = spark.table(silver_table)

total = silver_df.count()
unique = silver_df.select("order_id").distinct().count()

print("Silver rows:", total)
print("Unique orders:", unique)

assert total >= 8
assert total == unique

print("Silver task validation passed")